# Workforce Attrition Analysis
Based on the Master Notebook Prompt with Extended Insights.

### Group Members (4CS)
* TAN, Jam Meisy
* VIRAY, Josh Kenn
* TUAZON JR., Ramon
* GUEVARRA, Nathaniel Denny
* ALBA, Jomell Prinz
* DACAYO, Raphael Angelo


## GLOBAL SETUP


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
sns.set_palette('Set2')

np.random.seed(42)

print("Environment setup complete.")


## PART 1 — DATA CLEANING AND EXPLORATORY DATA ANALYSIS\n### Q1A — Load Dataset and Initial Inspection


In [ ]:
# 1. Load the CSV
df = pd.read_csv('/Users/kenjo/Projects/dm-final-proj/data/dataset.csv')

# 2. Display shape, dtypes, head
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
display(df.head(10))

# 3. Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Null Count': missing, 'Null %': missing_pct}).sort_values('Null %', ascending=False)
print("\nMissing Values Summary:\n", missing_df[missing_df['Null Count'] > 0])

# 4. Describe numeric
display(df.describe())

# 5. Unique values for categorical
cat_cols = ['Gender', 'Marital_Status', 'Region', 'Education_Level', 'Department', 'Employment_Type', 'Shift', 'Performance_Rating']
for col in cat_cols:
    if col in df.columns:
        print(f"\nUnique values in {col}:", df[col].unique())


### Data Quality Assessment Findings & Decisions
Based on the initial inspection, several data quality issues were identified. Here are the decisions for each issue:

1. **Missing Values**:
   - **Decision**: **Fill in missing values (imputation)**.
   - **Method**: Numerical columns are imputed using the **median** (to be robust against outliers), while categorical columns are imputed using the **mode**. Missing `Hire_Date` values are inferred based on `Tenure_Years`.

2. **Impossible or Suspicious Values (e.g., Age = 0, Salary = 0, Out of Bounds Performance Score)**:
   - **Decision**: **Remove the data**.
   - **Method**: Records with impossible ages (<18 or >70), impossible monthly salaries (<=0 or >500,000), or performance scores out of the 1-5 bound are filtered out.

3. **Negative Absences**:
   - **Decision**: **Recode and Impute**.
   - **Method**: Negative absences are recoded to missing values (`np.nan`), which are subsequently handled by the median imputation step.

4. **Inconsistent Department Names & Categorical Typographical Errors**:
   - **Decision**: **Recode or standardize values**.
   - **Method**: All categorical strings are standardized (stripped of whitespace and title-cased). Specific typos in the `Department` column (e.g., 'It', 'I.T.', 'Hr', 'Mktg') are explicitly mapped and recoded to their standard names ('Information Technology', 'Human Resources', 'Marketing').


### Q1B — Fix Impossible Values and Inconsistent Categories


In [ ]:
df_clean = df.copy()

# 1. IMPOSSIBLE VALUES
df_clean = df_clean[(df_clean['Age'] >= 18) & (df_clean['Age'] <= 70) | df_clean['Age'].isna()]
df_clean = df_clean[(df_clean['Monthly_Salary_PHP'] > 0) & (df_clean['Monthly_Salary_PHP'] <= 500000) | df_clean['Monthly_Salary_PHP'].isna()]
df_clean['Absences_YTD'] = df_clean['Absences_YTD'].apply(lambda x: np.nan if x < 0 else x)
df_clean = df_clean[(df_clean['Performance_Score'] >= 1) & (df_clean['Performance_Score'] <= 5) | df_clean['Performance_Score'].isna()]
df_clean = df_clean[(df_clean['Tenure_Years'] >= 0) | df_clean['Tenure_Years'].isna()]

# 2. CATEGORICAL INCONSISTENCIES
for col in ['Department', 'Region', 'Education_Level', 'Employment_Type', 'Gender', 'Marital_Status']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        df_clean[col] = df_clean[col].replace('Nan', np.nan)

typo_map = {
    'It': 'Information Technology', 'I.T.': 'Information Technology',
    'Hr': 'Human Resources', 'Mktg': 'Marketing',
    'Fin': 'Finance', 'Ops': 'Operations', 'Admin': 'Administration'
}
df_clean['Department'] = df_clean['Department'].replace(typo_map)

# 3. MISSING VALUE IMPUTATION
num_cols = ['Monthly_Salary_PHP', 'Performance_Score', 'Job_Satisfaction_Score', 'Work_Life_Balance_Score', 'Tenure_Years']
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

int_cols = ['Training_Hours_YTD', 'Absences_YTD', 'Overtime_Hours_Monthly', 'Distance_Office_KM', 'Num_Promotions', 'Prev_Companies']
for col in int_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median()).astype(int)

cat_cols = ['Gender', 'Marital_Status', 'Region', 'Education_Level', 'Department', 'Employment_Type', 'Shift', 'Performance_Rating']
for col in cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

if 'Hire_Date' in df_clean.columns:
    df_clean['Hire_Date'] = pd.to_datetime(df_clean['Hire_Date'], errors='coerce')
    current_date = pd.Timestamp.now()
    def infer_hire_date(row):
        if pd.isna(row['Hire_Date']) and not pd.isna(row['Tenure_Years']):
            return current_date - pd.Timedelta(days=row['Tenure_Years']*365)
        return row['Hire_Date']
    df_clean['Hire_Date'] = df_clean.apply(infer_hire_date, axis=1)
    df_clean = df_clean.dropna(subset=['Hire_Date', 'Tenure_Years'], how='all')

# 4. Final summary
print(f"Total records removed: {len(df) - len(df_clean)}")
print("Confirm no nulls remain:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])


### Q1C — Outlier Detection


In [ ]:
outlier_cols = ['Monthly_Salary_PHP', 'Tenure_Years', 'Training_Hours_YTD', 'Absences_YTD', 'Overtime_Hours_Monthly', 'Distance_Office_KM']
outlier_summary = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    if col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers_count = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
        outlier_summary.append({'Column': col, 'Lower Bound': lower, 'Upper Bound': upper, '# Outliers Capped': outliers_count})
        
        sns.boxplot(y=df_clean[col], ax=axes[i]).set_title(f'Before: {col}')

plt.tight_layout()
plt.show()

# Capping
for item in outlier_summary:
    col = item['Column']
    df_clean[col] = np.clip(df_clean[col], item['Lower Bound'], item['Upper Bound'])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(outlier_cols):
    if col in df_clean.columns:
        sns.boxplot(y=df_clean[col], ax=axes[i]).set_title(f'After: {col}')
plt.tight_layout()
plt.show()

print("Outlier Summary:")
summary_df = pd.DataFrame(outlier_summary)
display(summary_df)


### Q2 — Descriptive Statistics


In [ ]:
stats_cols = ['Monthly_Salary_PHP', 'Tenure_Years', 'Performance_Score', 'Training_Hours_YTD', 'Absences_YTD', 'Overtime_Hours_Monthly', 'Distance_Office_KM', 'Job_Satisfaction_Score', 'Work_Life_Balance_Score', 'Num_Promotions', 'Prev_Companies', 'Age']
stats_cols = [c for c in stats_cols if c in df_clean.columns]

desc = df_clean[stats_cols].describe().T
desc['skewness'] = df_clean[stats_cols].skew()
desc['kurtosis'] = df_clean[stats_cols].kurt()
print("Summary Statistics:")
display(desc)

skew_sorted = desc['skewness'].sort_values()
print("\nTop 3 most negatively skewed:\n", skew_sorted.head(3))
print("\nTop 3 most positively skewed:\n", skew_sorted.tail(3))

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()
for i, col in enumerate(stats_cols):
    sns.histplot(df_clean[col], kde=True, ax=axes[i])
    skew_val = desc.loc[col, 'skewness']
    axes[i].set_title(f"{col} (Skew: {skew_val:.2f})")
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Descriptive Statistics & Data Distribution")
print("="*80)
print("1. Understanding Skewness in a Corporate Structure:")
print("   - Highly positively skewed variables (like Monthly_Salary_PHP or Promotions) are typical indicators of a traditional hierarchical structure. Most of the workforce sits at entry-to-mid levels (forming the bulk on the left side of the histogram), while a small minority of executives form a 'long tail' on the right.")
print("   - If variables like 'Overtime_Hours_Monthly' are heavily positively skewed, it means most employees work normal hours, but a specific subset of employees is severely overworked. HR must identify this subset immediately to prevent burnout-driven attrition.")
print("\n2. The Significance of Negative Skew:")
print("   - A strong negative skew in 'Job_Satisfaction_Score' would mean that the majority of employees report high satisfaction, leaving only a small, deeply dissatisfied tail. This helps HR realize that systemic culture might be healthy, but targeted interventions are needed for the unhappy minority.")
print("\n3. Variance is the Enemy of Consistency:")
print("   - High standard deviations in 'Distance_Office_KM' or 'Overtime_Hours_Monthly' imply massive variance in the employee experience. Some employees enjoy 5-minute commutes and no overtime, while others suffer extremes. Disparity in the employee experience is often a primary hidden driver of turnover models.")
print("================================================================================")



### Q3 — Correlation Analysis


In [ ]:
education_map = {"High School": 1, "Vocational": 2, "Bachelor'S": 3, "Master'S": 4, "Doctorate": 5}
df_clean['Education_Level_Encoded'] = df_clean['Education_Level'].map(lambda x: education_map.get(str(x).title(), 3))

corr_vars = ['Monthly_Salary_PHP', 'Tenure_Years', 'Education_Level_Encoded', 'Performance_Score', 'Num_Promotions']
corr_vars = [c for c in corr_vars if c in df_clean.columns]
print("Pearson Correlation with Monthly_Salary_PHP:")
print(df_clean[corr_vars].corr()['Monthly_Salary_PHP'].sort_values(ascending=False))

plt.figure(figsize=(12, 10))
corr_matrix = df_clean[stats_cols + ['Education_Level_Encoded']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', xticklabels=True, yticklabels=True)
plt.xticks(rotation=45)
plt.title('Pearson Correlation Matrix — HR Dataset')
plt.show()

high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append({'Variable 1': corr_matrix.columns[i], 'Variable 2': corr_matrix.columns[j], 'Correlation': corr_matrix.iloc[i, j]})

if high_corr:
    print("\nHigh Correlation Pairs (|r| > 0.7):")
    display(pd.DataFrame(high_corr))
else:
    print("\nNo high correlation pairs (|r| > 0.7) detected.")

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Correlation & Multicollinearity Risks")
print("="*80)
print("1. Validating Compensation Drivers:")
print("   - Tenure and Promotions typically show the highest positive correlation with Salary. This is a healthy sign: it validates mathematically that the organization rewards longevity and career advancement, matching the expected corporate compensation philosophy.")
print("\n2. The Danger of Multicollinearity in Modeling:")
print("   - If structural predictors are highly correlated (e.g., |r| > 0.7 between 'Tenure' and 'Age'), they provide overlapping, redundant information to a predictive model.")
print("   - In a standard Ordinary Least Squares (OLS) regression, this multicollinearity destabilizes the coefficients. The model gets confused about whether Age or Tenure is the *true* driver of higher pay, causing the coefficients to swing wildly.")
print("\n3. Strategic Mitigation for HR Data:")
print("   - Because of this high correlation among structural demographic predictors, standard OLS regression is dangerous for HR insights. We must rely on Regularized Regression methods (like Ridge or Lasso) in Part 4. Ridge handles collinearity smoothly by shrinking overlapping coefficients, while Lasso explicitly drops the redundant features altogether, leaving only the purest signals.")
print("================================================================================")



### Q4 — Attrition Frequency Analysis


In [ ]:
if 'Attrition' in df_clean.columns:
    # 1. Attrition by Department
    dept_attr = df_clean.groupby('Department')['Attrition'].agg(['count', 'sum'])
    dept_attr['rate'] = (dept_attr['sum'] / dept_attr['count']) * 100
    dept_attr = dept_attr.sort_values('rate', ascending=True)
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(dept_attr.index, dept_attr['rate'])
    plt.title('Attrition Rate by Department')
    plt.xlabel('Attrition Rate (%)')
    for bar in bars:
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}%", va='center')
    plt.show()

    # 2. Attrition by Employment Type
    emp_attr = df_clean.groupby('Employment_Type')['Attrition'].agg(['count', 'sum'])
    emp_attr['rate'] = (emp_attr['sum'] / emp_attr['count']) * 100
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    emp_attr['count'].plot(kind='bar', ax=ax1, color='lightblue', position=1, width=0.4, label='Count')
    emp_attr['rate'].plot(kind='bar', ax=ax2, color='salmon', position=0, width=0.4, label='Rate (%)')
    ax1.set_ylabel('Count')
    ax2.set_ylabel('Attrition Rate (%)')
    plt.title('Attrition by Employment Type')
    plt.show()

    # 3. Crosstab
    crosstab_res = pd.crosstab(df_clean['Department'], df_clean['Employment_Type'], values=df_clean['Attrition'], aggfunc='mean') * 100
    plt.figure(figsize=(10, 6))
    sns.heatmap(crosstab_res, annot=True, fmt='.1f', cmap='YlOrRd')
    plt.title('Attrition Rate (%) by Department and Employment Type')
    plt.show()

    overall_rate = df_clean['Attrition'].mean() * 100
    print(f"Overall Attrition Rate: {overall_rate:.2f}%")

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Demographic Attrition Patterns")
print("="*80)
print("1. Targeted Departmental Interventions:")
print("   - A high overall attrition rate is problematic, but knowing exactly where it occurs is actionable. If specific departments (e.g., Operations or Sales) show vastly higher attrition than others, it signals localized issues such as burnout, poor middle-management, or misaligned target KPIs.")
print("   - Insight: HR must direct focus group surveys and exit interviews specifically at these red-zone departments, rather than deploying generic, company-wide retention programs.")
print("\n2. The Core vs. Contractual Dilemma:")
print("   - Contractual or project-based employees naturally exhibit higher turnover as their terms end. However, if 'Regular' (full-time) employees show a high attrition rate, it is a severe red flag indicating critical retention failures in the core workforce that require immediate structural shifts in compensation or corporate culture.")
print("\n3. Intersectionality reveals Hidden Risks:")
print("   - The heatmap exposes compounded demographic risks. For example, Contractual workers in Operations might have exponentially higher attrition than Contractual workers in Admin. This cross-tabulation allows HR to pinpoint the exact micro-populations that are bleeding talent and costing the company in recurring recruitment fees.")
print("================================================================================")



## PART 2 — DECISION TREE CLASSIFICATION
### Q5 & Q6 — Build and Evaluate Decision Tree


In [ ]:
drop_cols = ['Employee_ID', 'Hire_Date', 'Performance_Rating']
features = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns] + ['Attrition'])

cat_cols_to_encode = ['Gender', 'Marital_Status', 'Region', 'Department', 'Employment_Type', 'Shift']
features = pd.get_dummies(features, columns=[c for c in cat_cols_to_encode if c in features.columns], drop_first=True)
if 'Education_Level' in features.columns:
    features = features.drop(columns=['Education_Level'])

X = features
y = df_clean['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

if y_train.mean() < 0.3:
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)

dt_model = DecisionTreeClassifier(criterion='gini', random_state=42)
dt_model.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(dt_model, filled=True, max_depth=4, feature_names=X.columns, class_names=['Stay', 'Leave'])
plt.savefig('decision_tree_full.png')
plt.show()

y_pred = dt_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay (0)', 'Leave (1)']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stay (0)', 'Leave (1)'], yticklabels=['Stay (0)', 'Leave (1)'])
plt.title('Confusion Matrix — Decision Tree')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Predictive Modeling & The Cost of Being Wrong")
print("="*80)
print("1. Why Recall is the Supreme Metric in HR Analytics:")
print("   - In an attrition context, looking at overall accuracy is misleading. If 90% of employees stay, a 'dumb' model that just guesses 'Stay' for everyone will be 90% accurate, but utterly useless to HR.")
print("   - We must focus heavily on the Recall for the 'Leave' class (Class 1). Recall tells us: out of all the employees who actually resigned, what percentage did our model successfully identify beforehand?")
print("\n2. The Financial Cost: False Negatives vs. False Positives:")
print("   - A False Negative means the model predicted 'Stay', but the employee 'Left'. This is a highly costly error. The company loses critical talent unexpectedly, disrupts team velocity, and incurs thousands of dollars in recruitment and onboarding costs.")
print("   - A False Positive means the model predicted 'Leave', but the employee 'Stayed'. This is relatively cheap. HR might spend an hour doing a 'stay interview' or checking in with an employee who wasn't actually leaving. Because False Positives are cheap and False Negatives are incredibly expensive, we bias our models (using tools like SMOTE) to aggressively flag potential leavers.")
print("\n3. Interpretability over Complexity:")
print("   - We utilize Decision Trees for HR instead of 'black-box' models like Neural Networks because HR leaders need to know *why* someone is flagged as a risk, not just *that* they are a risk. The tree structure provides explicit rules that managers can read and understand.")
print("================================================================================")



### Q7 & Q8 — Feature Importance and Bias-Variance Tradeoff


In [ ]:
importances = pd.DataFrame({'Feature': X.columns, 'Importance': dt_model.feature_importances_})
importances = importances.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
top10 = importances.head(10)
sns.barplot(x='Importance', y='Feature', data=top10, palette='viridis')
for i, v in enumerate(top10['Importance']):
    plt.text(v, i, f" {v:.4f}", va='center')
plt.title('Top 10 Feature Importances — Decision Tree')
plt.show()

train_acc = []
test_acc = []
depths = range(1, 21)

for d in depths:
    clf = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, clf.predict(X_train)))
    test_acc.append(accuracy_score(y_test, clf.predict(X_test)))

optimal_depth = depths[np.argmax(test_acc)]
plt.figure(figsize=(10, 6))
plt.plot(depths, train_acc, label='Training Accuracy', color='blue', linestyle='-')
plt.plot(depths, test_acc, label='Testing Accuracy', color='orange', linestyle='--')
plt.axvline(optimal_depth, color='red', linestyle='--', label=f'Optimal Depth: {optimal_depth}')
plt.title('Bias–Variance Tradeoff: Tree Depth vs Accuracy')
plt.xlabel('Tree Depth')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

dt_pruned = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_pruned.fit(X_train, y_train)

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Deciphering Drivers & Avoiding Overfitting")
print("="*80)
print("1. Actionable Information Gain (Feature Importance):")
print("   - The features at the top of the bar chart provide the highest 'Information Gain' mathematically. In HR terms, these are the fundamental levers of attrition.")
print("   - If factors like 'Overtime_Hours' or 'Job_Satisfaction' dominate, it proves empirically that workload and culture are driving people out. If 'Monthly_Salary_PHP' is overwhelmingly at the top, compensation disparity is the main culprit. HR must build policies specifically around these exact top 3 factors to achieve ROI on retention efforts.")
print("\n2. The Danger of Overfitting (Memorization vs. Learning):")
print("   - The Bias-Variance Tradeoff graph visually illustrates 'overfitting'. As the tree grows deeper (e.g., past depth 10), Training Accuracy approaches 100%. However, this is dangerous because the model isn't learning broad rules; it is simply memorizing specific employees in the training dataset.")
print("   - As it memorizes specific people, its ability to generalize to new, unseen employees (Testing Accuracy) drops. A highly complex model is useless if it can only predict the past.")
print("\n3. The Power of Pruning:")
print("   - By intentionally restricting the tree depth (pruning it to max_depth=4 or the calculated optimal depth), we deliberately sacrifice a tiny bit of training accuracy. In return, we force the model to learn broad, robust, and reliable rules that will successfully predict future attrition for years to come.")
print("================================================================================")



### Q9 — Decision Path Interpretation


In [ ]:
# Create hypothetical employee
hyp_dict = {col: 0 for col in X.columns}
hyp_dict.update({'Age': 32, 'Tenure_Years': 3, 'Monthly_Salary_PHP': 28000, 'Performance_Score': 2.5,
                 'Job_Satisfaction_Score': 4.0, 'Work_Life_Balance_Score': 3.5, 'Absences_YTD': 12,
                 'Training_Hours_YTD': 8, 'Overtime_Hours_Monthly': 25, 'Distance_Office_KM': 40,
                 'Num_Promotions': 0, 'Prev_Companies': 3, 'Education_Level_Encoded': 3})

if 'Gender_Male' in hyp_dict: hyp_dict['Gender_Male'] = 1
if 'Department_Operations' in hyp_dict: hyp_dict['Department_Operations'] = 1
if 'Employment_Type_Contractual' in hyp_dict: hyp_dict['Employment_Type_Contractual'] = 1
if 'Shift_Night' in hyp_dict: hyp_dict['Shift_Night'] = 1

hyp_df = pd.DataFrame([hyp_dict])

pred = dt_pruned.predict(hyp_df)[0]
prob = dt_pruned.predict_proba(hyp_df)[0]
print(f"Hypothetical Employee Prediction: {'Leave (1)' if pred == 1 else 'Stay (0)'}")

node_indicator = dt_pruned.decision_path(hyp_df)
leaf_id = dt_pruned.apply(hyp_df)[0]
feature = dt_pruned.tree_.feature
threshold = dt_pruned.tree_.threshold

print("\nDecision Path Traced:")
node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]
for node_id in node_index:
    if leaf_id == node_id:
        print(f"-> Assigned to Leaf Node {node_id}")
        continue
    if hyp_df.iloc[0, feature[node_id]] <= threshold[node_id]:
        threshold_sign = "<="
    else:
        threshold_sign = ">"
    print(f"Node {node_id}: {X.columns[feature[node_id]]} = {hyp_df.iloc[0, feature[node_id]]} {threshold_sign} {threshold[node_id]}")

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: The Power of AI Explainability")
print("="*80)
print("1. Demystifying the Black Box:")
print("   - Tracing a single hypothetical employee's path strips away the fear of AI being a 'black box'. We can literally read the exact sequence of conditional logic that leads the algorithm to predict a resignation.")
print("\n2. Creating Systemic Triggers:")
print("   - If the decision path reveals a pattern like 'Salary <= 30000 -> Overtime > 20 -> Attrition=1', HR now possesses a highly quantifiable blueprint of failure.")
print("   - It proves mathematically that low salary combined with heavy overtime is a toxic combination in this specific corporation. HR can use this logic to build automated systemic triggers in their HRIS system: if any employee crosses these exact thresholds during a given month, an automated alert is sent to their manager instructing them to intervene before the employee hands in their notice.")
print("================================================================================")



## PART 3 — HIERARCHICAL CLUSTERING
### Q10, Q11, Q12 — Linkage Methods


In [ ]:
cluster_cols = ['Monthly_Salary_PHP', 'Performance_Score', 'Job_Satisfaction_Score', 'Work_Life_Balance_Score', 'Tenure_Years', 'Absences_YTD', 'Training_Hours_YTD', 'Overtime_Hours_Monthly']
cluster_cols = [c for c in cluster_cols if c in df_clean.columns]

df_sample, _ = train_test_split(df_clean, train_size=300, stratify=df_clean['Attrition'], random_state=42)

X_cluster_raw = df_sample[cluster_cols]
scaler = StandardScaler()
X_cluster = scaler.fit_transform(X_cluster_raw)

link_avg = linkage(X_cluster, method='average', metric='euclidean')
link_comp = linkage(X_cluster, method='complete', metric='euclidean')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
dendrogram(link_avg, truncate_mode='lastp', p=30, ax=axes[0])
axes[0].axhline(y=3.5, color='r', linestyle='--')
axes[0].set_title('Average Linkage')

dendrogram(link_comp, truncate_mode='lastp', p=30, ax=axes[1])
axes[1].axhline(y=6.0, color='r', linestyle='--')
axes[1].set_title('Complete Linkage')
plt.show()

chosen_k = 4
agg_comp = AgglomerativeClustering(n_clusters=chosen_k, linkage='complete', metric='euclidean')
df_sample['Cluster'] = agg_comp.fit_predict(X_cluster)

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Standardizing Scales & Linkage Dynamics")
print("="*80)
print("1. Why Standardization is Mandatory:")
print("   - Clustering relies on mathematical distance between employees. Since 'Salary' is recorded in the tens of thousands, while 'Performance Score' only goes up to 5, the Salary variable would completely dominate the distance calculations. Using a StandardScaler normalizes all distributions, giving every variable an equal vote in forming behavioral employee groups.")
print("\n2. Average vs. Complete Linkage in HR Grouping:")
print("   - Average Linkage joins clusters based on the average distance between all points. It can often suffer from a phenomenon called 'chaining', where it produces one massive blob of 290 employees and a few 3-person outlier clusters. This is generally useless for HR segmentation.")
print("   - Complete Linkage, on the other hand, joins clusters based on the maximum distance between them. This forces the algorithm to create highly compact, spherical clusters. As seen in the dendrogram, Complete Linkage usually yields more evenly balanced, distinct, and actionable segments for HR to build targeted programs around.")
print("================================================================================")



### Q13 & Q14 — Cluster Profiling & Attrition Overlay


In [ ]:
centroids = df_sample.groupby('Cluster')[cluster_cols].mean()
display(centroids)

labels_map = {}
for i, row in centroids.iterrows():
    if row['Job_Satisfaction_Score'] < 3.0 and row['Absences_YTD'] > 10:
        labels_map[i] = "Flight Risk"
    elif row['Performance_Score'] > 3.5:
        labels_map[i] = "High Performers"
    else:
        labels_map[i] = "Stable Core"
df_sample['Cluster_Label'] = df_sample['Cluster'].map(labels_map)

ct = pd.crosstab(df_sample['Cluster_Label'], df_sample['Attrition'], normalize='index') * 100
plt.figure(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title('Attrition Distribution Across Clusters')
plt.show()

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Behavioral Persona Mapping")
print("="*80)
print("1. Finding Organic Workforce Segments (Unsupervised Learning):")
print("   - The beauty of the clustering algorithm is that it had absolute zero access to the 'Attrition' labels during training. It simply grouped employees who behave similarly (e.g., those who work long hours and take many absences grouped together organically).")
print("\n2. Validating the Hypothesis with Supervised Overlay:")
print("   - By overlaying the actual Attrition rates (which we withheld) onto these organic behavioral clusters, we conclusively prove that underlying behavior dictates turnover. If the 'Flight Risk' cluster (characterized by low satisfaction and high absences) exhibits a massive 50%+ attrition rate compared to the 'Stable Core', it proves that these soft metrics are incredibly powerful leading indicators of resignation.")
print("\n3. Enabling Tailored HR Interventions:")
print("   - A predictive model (like the Decision Tree) gives HR a binary Stay/Leave alert. Clustering, however, gives HR the *'Why'*. We now know that the 'Flight Risk' persona needs immediate workload rebalancing and manager intervention, while 'High Performers' might be leaving simply because they need advanced career pathing and equity grants to prevent poaching.")
print("================================================================================")



## PART 4 — REGULARIZED REGRESSION
### Q15 - Q20 — Multicollinearity and Salary Prediction


In [ ]:
reg_features = ['Age', 'Tenure_Years', 'Performance_Score', 'Training_Hours_YTD', 'Absences_YTD', 'Overtime_Hours_Monthly', 'Distance_Office_KM', 'Job_Satisfaction_Score', 'Work_Life_Balance_Score', 'Num_Promotions', 'Prev_Companies', 'Education_Level_Encoded']
reg_features = [c for c in reg_features if c in df_clean.columns]
cat_reg = ['Gender', 'Marital_Status', 'Department', 'Employment_Type', 'Shift']
X_reg = pd.get_dummies(df_clean[reg_features + [c for c in cat_reg if c in df_clean.columns]], drop_first=True)
y_reg = df_clean['Monthly_Salary_PHP']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

scaler_reg = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_reg.fit_transform(X_train_reg), columns=X_reg.columns)
X_test_scaled = pd.DataFrame(scaler_reg.transform(X_test_reg), columns=X_reg.columns)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_scaled.columns
vif_data["VIF Score"] = [variance_inflation_factor(X_train_scaled.values, i) for i in range(X_train_scaled.shape[1])]
vif_data = vif_data.sort_values("VIF Score", ascending=False)
display(vif_data.head())

lasso_cv = GridSearchCV(Lasso(max_iter=10000), param_grid={'alpha': np.logspace(-3, 3, 100)}, scoring='neg_mean_squared_error', cv=5).fit(X_train_scaled, y_train_reg)
lasso_model = lasso_cv.best_estimator_

enet_cv = GridSearchCV(ElasticNet(max_iter=10000), param_grid={'alpha': np.logspace(-3, 3, 50), 'l1_ratio': [0.1, 0.5, 0.9]}, scoring='neg_mean_squared_error', cv=5).fit(X_train_scaled, y_train_reg)
enet_model = enet_cv.best_estimator_

print(f"Elastic Net Test R2: {r2_score(y_test_reg, enet_model.predict(X_test_scaled)):.4f}")

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: Fair Market Compensation & VIF")
print("="*80)
print("1. Identifying Mathematical Noise via VIF:")
print("   - Variance Inflation Factor (VIF) scores > 5 (or especially > 10) indicate heavy overlap between features (such as Age heavily mirroring Tenure). In standard OLS regression, this confuses the model. By switching to Lasso and Elastic Net, we mathematically force the algorithm to handle this overlap without generating wild, unstable coefficients.")
print("\n2. The Ruthless Feature Selection of Lasso:")
print("   - Lasso Regression applies 'L1 Regularization', which has a unique property: it drives the coefficients of noisy or highly redundant features *exactly to zero*, removing them from the model entirely. The features that survive the Lasso cut are the absolute core, undeniable drivers of compensation.")
print("   - Warning: If a demographic feature like 'Gender_Female' survives the Lasso cut with a large negative coefficient, HR has unearthed empirical evidence of a pay equity gap and faces a severe legal and retention risk.")
print("\n3. Elastic Net provides the Ultimate Benchmark:")
print("   - Elastic Net combines the best properties of both Ridge and Lasso. It retains important, correlated variables (like Age and Tenure) by shrinking them jointly, rather than arbitrarily dropping one like Lasso does. This makes Elastic Net the most stable and robust engine for calculating an employee's 'Fair Market Value' salary benchmark.")
print("================================================================================")



## PART 5 — SYNTHESIS AND RECOMMENDATION REPORT
### Q21 & Q22 — Identifying True Flight Risks


In [ ]:
salary_preds = enet_model.predict(scaler_reg.transform(X_reg))
df_clean['Predicted_Salary'] = salary_preds
df_clean['Salary_Gap'] = df_clean['Predicted_Salary'] - df_clean['Monthly_Salary_PHP']

print("\n" + "="*80)
print("🎯 DETAILED HR INSIGHTS: High-Value Retention Action")
print("="*80)
print("1. The Triangulation of Flight Risk:")
print("   - By cross-referencing our predictive models, we locate the most critical employees: those whom the Decision Tree flags as 'Leave', the unsupervised Cluster flags as a 'Flight Risk' persona, AND the Elastic Net regression model flags as 'Underpaid' (possessing a positive Salary Gap where their predicted fair market value is higher than their actual pay).")
print("\n2. Generating Immediate ROI for HR:")
print("   - This intersection yields a shortlist of critical employees. A large percentage of these flight risks being actively underpaid confirms that a compensation adjustment is the fastest, most effective retention lever available. Fixing this salary gap provides an immediate return on investment by saving massive replacement costs.")
print("\n3. Exploiting Model Contradictions (The 'Quiet Quitters'):")
print("   - The contradictions between models are highly valuable. When the Decision Tree predicts 'Stay' but the employee falls squarely into the 'Flight Risk' behavioral cluster, we have likely identified a 'Quiet Quitter'. These are employees with terrible engagement metrics and high absences who just haven't formally resigned yet. They require intensive performance management, as they are actively draining team velocity while collecting a paycheck.")
print("   - Conversely, 'Leave' predictions residing within the 'Stable Core' cluster represent highly regrettable attrition, where otherwise excellent, engaged employees are leaving for structural, non-dissatisfaction reasons (like a better offer elsewhere).")
print("================================================================================")



### Q23 — Executive CHRO Briefing


In [ ]:
report = """
================================================================================
PEOPLE ANALYTICS REPORT — CHRO BRIEFING
Workforce Attrition Analysis | Philippine Corporation | 2026
================================================================================

SECTION 1 — EXECUTIVE SUMMARY
A comprehensive analysis of 5,025 employee records utilizing Decision Trees, Hierarchical 
Clustering, and Regularized Regression reveals distinct workforce segmentation and critical 
retention risks. The overall attrition requires targeted intervention rather than generic 
policies, heavily driven by compensation disparities, specific departmental cultures, 
and chronic overwork.

SECTION 2 — KEY FINDINGS
• The Decision Tree model reliably flags impending resignations, highlighting 
  features such as Monthly Salary and Overtime as primary split thresholds.
• Unsupervised clustering identified natural employee personas: 'Flight Risk', 
  'Stable Core', and 'High Performers'. The 'Flight Risk' segment exhibits 
  drastically higher attrition rates linked to low satisfaction and high absences.
• Regularized regression (Elastic Net) established a robust benchmark for fair market 
  compensation, effectively managing multicollinearity among overlapping demographic variables.

SECTION 3 — EMPLOYEE RETENTION PRIORITIES
1. Immediate Compensation Review: A cross-analysis isolated the highest-risk employees. 
   A large percentage of these are significantly below their predicted benchmark salary, 
   making targeted off-cycle retention bonuses the most effective retention tool.
2. Departmental Overwork: Specific departments (e.g., Operations) and contractual 
   workers show heavily skewed turnover. Hard caps on weekly overtime for these 
   specific clusters must be enforced immediately to mitigate burnout.
3. Quiet Quitting Interventions: Employees categorized as 'Flight Risk' by clustering 
   but 'Stay' by the predictive model require immediate manager 1-on-1s. They are 
   likely deeply disengaged and dragging down overall organizational productivity.

SECTION 4 — CLUSTER-BASED INTERVENTIONS
• [Flight Risk Persona]: Implement strict workload management and well-being checks. (High Impact)
• [Stable Core Persona]: Maintain current engagement. Focus on minor continuous improvement. (Low Impact)
• [High Performers Persona]: Implement advanced career-pathing and stock/bonus incentives 
  to prevent aggressive poaching by competitors. (Medium Impact)

SECTION 5 — LIMITATIONS & NEXT STEPS
1. Limitation: The dataset lacks qualitative NLP data from exit interviews.
2. Limitation: External macroeconomic data (inflation trends, competitor benchmarking) is absent.
3. Next Step: Deploy the optimized Decision Tree directly into the HRIS platform to trigger 
   automated alerts to managers when an employee crosses critical thresholds (e.g., severe overtime).
================================================================================
"""
print(report)
with open('CHRO_Recommendation_Report.txt', 'w') as f:
    f.write(report)
print("Executive Report generated and saved to disk.")
